# EyeLink ASC Parser and Pupil Size PCHIP Interpolation (Google Colab Version)
# EyeLink ASCパーサー & 瞳孔径PCHIP補間 (Google Colab版)

This notebook provides a complete, self-contained pipeline for parsing raw EyeLink `.asc` files and cleaning the resulting pupil size data using PCHIP interpolation to bridge blink artifacts.

このノートブックは、生のエレック `.asc` ファイルを解析し、瞬きによるアーティファクトをPCHIP補間によって滑らかに修正・クリーニングするパイプラインを提供します。

### Pipeline Overview / パイプラインの手順:
1. **File Upload**: Upload your `.asc` file exported from EyeLink.
   - **ファイルのアップロード**: EyeLinkからエクスポートした `.asc` ファイルをアップロードします。
2. **Parsing**: Reconstructs sample timestamps, elapsed times, and the binary stimulus activity indicator (`blue_active`) derived from Eyelink `BUTTON` and `INPUT` events.
   - **パース（解析）**: サンプルのタイムスタンプ、経過時間、および Eyelink の `BUTTON`/`INPUT` イベントから取得した刺激アクティブインジケータ（`blue_active`）を再構築します。
3. **Data Cleaning**: Identifies blinks (dropouts where pupil tracking is lost), expands the blink boundary window to clean eyelid closing/opening transients, and performs **PCHIP (Piecewise Cubic Hermite Interpolating Polynomial)** interpolation to smoothly reconstruct the missing data.
   - **データクリーニング**: 瞬き（瞳孔径が極端に低下した箇所）を検出し、瞼の開閉部分を完全に補間するためにマスク枠を少し前後に拡張した上で、**PCHIP（区分的3次エルミート補間多項式）**を用いて欠損データを滑らかに繋ぎます。
4. **Visualization**: Plots raw vs. cleaned pupil size, highlighting when the blue LED stimulus was active.
   - **視覚化**: 生データとクリーニングした瞳孔径の重ね合わせグラフを表示し、青色LED刺激がONだった時間帯を明示します。
5. **File Download**: Exports the final processed dataset as CSV and Excel files and downloads them directly to your computer.
   - **ファイルのダウンロード**: 処理したデータをCSVおよびExcel形式で書き出し、自動的にダウンロードします。

In [ ]:
!pip install -q japanize-matplotlib

In [ ]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

try:
    from google.colab import files
    IN_COLAB = True
    print("Running on Google Colab. Ready to upload/download files!")
    print("Google Colabで実行中。ファイルのアップロード/ダウンロードの準備が完了しました！")

except ImportError:
    IN_COLAB = False
    print("Running locally. Local mode enabled!")
    print("ローカル環境で実行中。ローカルモードが有効になりました！")


## Step 1: Upload the EyeLink `.asc` File
## ステップ 1: EyeLink `.asc` ファイルのアップロード

Run the cell below. A file upload box will appear. Select your `.asc` file.

下のセルを実行し、表示されるアップロードボックスから `.asc` ファイルを選択してください。

In [ ]:
if IN_COLAB:
    print("Please upload your Eyelink .asc file:")
    print("Eyelinkの .asc ファイルをアップロードしてください:")
    uploaded = files.upload()
    uploaded_filenames = list(uploaded.keys())

    if len(uploaded_filenames) > 0:
        asc_filename = uploaded_filenames[0]
        print(f"\nSuccessfully uploaded: {asc_filename}")
        print(f"アップロード完了: {asc_filename}")
    else:
        raise ValueError("No file was uploaded. Please re-run this cell and upload a .asc file. / ファイルがアップロードされませんでした。セルを再実行してください。")

else:
    
    asc_filename = input('ASC filepath: ')
    
    print(f"Running locally. Targeting file: {asc_filename}")
    print(f"ローカル環境で実行中。対象ファイル: {asc_filename}")
    
    if not os.path.exists(asc_filename):
        raise FileNotFoundError(
            f"Could not find Eyelink .asc file at path: '{asc_filename}'. "
            f"Please update the path in this cell or ensure the file exists."
        )


## Step 2: Define and Run the Parsing & Interpolation Pipeline
## ステップ 2: 解析 & 補間パイプラインの定義と実行

### 💡 Note on Adjusting the Sliding Window Size (`SLIDING_WINDOW_SIZE`)
### 💡 スライディングウィンドウサイズ (`SLIDING_WINDOW_SIZE`) の調整に関する注意点

The sliding window size is used to expand the blink detection mask. Because the eyelid partially covers the pupil immediately before and after a blink, the eye tracker records distorted/bad data at the edges of a blink.

スライディングウィンドウは、瞬き検出マスクを拡張するために使用されます。瞬きの直前および直後は、瞼が部分的に瞳孔を覆うため、アイトラッカーは瞬きのエッジ部分で歪んだ不良データを記録します。

- **At 1000 Hz (1 ms per sample)**: A sliding window size of `200` (representing 200 ms total, or 100 ms before and after the center of the blink) is recommended to cover the eyelid closing and opening phases.
  - **1000 Hz（1ミリ秒に1サンプル）の場合**: 瞼の閉じる相と開く相を完全にカバーするために、スライディングウィンドウサイズ `200`（総計200 ms、すなわち瞬きの中心から前後 100 ms ずつ）が推奨されます。
- **Adjusting for your data**: If you notice the edges of blinks are not being fully cleaned and still drop down toward zero, increase `SLIDING_WINDOW_SIZE` (e.g., to `300`). If valid data points are being cut off unnecessarily, decrease this value (e.g., to `100` or `150`) in the configuration block below.
  - **データに合わせた調整**: もし瞬き前後の歪みデータが十分にクリーニングされずに0に向かって急降下している部分が残っている場合は、`SLIDING_WINDOW_SIZE` の値を大きく（例: `300`）してください。逆に、正常なデータ点まで不必要にカットされてしまっている場合は、値を小さく（例: `100` または `150`）調整してください。

In [ ]:
# ==============================================================================
# PARSING & INTERPOLATION CONFIGURATION / 解析・補間パラメータ設定
# ==============================================================================

# 1. Sliding window size (ms) to expand the blink detection mask
# 瞬き検出マスクを拡張するためのスライディングウィンドウサイズ（ミリ秒）
SLIDING_WINDOW_SIZE = 200

# 2. Pupil size threshold below which data is treated as a blink/signal loss
# この値未満の瞳孔径データを瞬きまたは信号消失として扱います
BLINK_THRESHOLD = 500

# 3. Pre-saccade padding duration (ms) to mask before the detected saccade onset
# サッケード検出開始の前にマスクするプリパッド時間（ミリ秒）
SACCADE_PRE_PAD_MS = 20

# 4. Post-saccade padding duration (ms) to mask after the detected saccade offset
# サッケード検出終了の後にマスクするポストパッド時間（ミリ秒）
SACCADE_POST_PAD_MS = 60

# 5. Default flash duration (ms) used to reconstruct button events when OFF is missing
# OFFイベントが欠落している場合にボタンイベントを再構成するためのデフォルトのフラッシュ時間（ミリ秒）
DEFAULT_FLASH_DURATION_MS = 750


In [ ]:
def parse_and_clean_asc(file_source, filename_label):
    """
    Parses an EyeLink .asc file from Google Colab memory,
    computes elapsed times, maps button/input events to blue_active state, and cleans pupil data
    using PCHIP interpolation over blinks and saccades.
    """
    timestamps = []
    pupil_size = []
    button_events = []  # List of tuples: (btn_timestamp, event_type, state)
    saccade_events = [] # List of tuples: (eye, start_timestamp, end_timestamp)
    tracked_eye = "L"   # Default to Left eye
    in_recording = False
    
    # 1. Parse File Content
    content = file_source.read()
    # Decode if binary content
    if isinstance(content, bytes):
        lines = content.decode("utf-8", errors="ignore").splitlines()
    else:
        lines = content.splitlines()
            
    print(f"Analyzing {len(lines)} lines from {filename_label}...")
    print(f"{filename_label} から {len(lines)} 行を解析中...")
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        parts = line.split()
        first_word = parts[0]
        
        if first_word == "START":
            in_recording = True
            if len(parts) >= 3:
                if "LEFT" in parts:
                    tracked_eye = "L"
                elif "RIGHT" in parts:
                    tracked_eye = "R"
            continue
        elif first_word == "END":
            in_recording = False
            continue
            
        if not in_recording:
            continue
        
        # Discard eye tracking event lines explicitly to prevent disruption in sample/timestamp parsing
        if first_word in {"EFIX", "SFIX", "EBLINK", "SBLINK", "MSG", "PRESCALER", "VPRESCALER", "PUPIL", "EVENTS", "SAMPLES"}:
            continue
        
        # Check if sample line (starts with numeric timestamp)
        try:
            ts = int(first_word)
            timestamps.append(ts)
            
            # Pupil Size is the 4th column (index 3) in EyeLink ASC sample lines
            if len(parts) > 3:
                pupil_size.append(float(parts[3]))
            else:
                pupil_size.append(0.0)
                
        except ValueError:
            # Event Line (BUTTON or INPUT)
            if first_word == "BUTTON" and len(parts) >= 4:
                try:
                    btn_ts = int(parts[1])
                    btn_state = int(parts[3])
                    button_events.append((btn_ts, "BUTTON", btn_state))
                except ValueError:
                    pass
            elif first_word == "INPUT" and len(parts) >= 3:
                try:
                    inp_ts = int(parts[1])
                    inp_val = int(parts[2])
                    # Treat 127 as off (0), other inputs (like 63) as on (1)
                    inp_state = 0 if inp_val == 127 else 1
                    button_events.append((inp_ts, "INPUT", inp_state))
                except ValueError:
                    pass
            elif (first_word == "SSACC" or first_word == "ESACC") and len(parts) >= 3:
                try:
                    eye = parts[1]
                    start_ts = int(parts[2])
                    if first_word == "ESACC" and len(parts) >= 4:
                        end_ts = int(parts[3])
                    else:
                        end_ts = start_ts
                    saccade_events.append((eye, start_ts, end_ts))
                except ValueError:
                    pass

    # Sort and Deduplicate button/input events at identical timestamps (retaining/prioritizing BUTTON states if they overlap)
    button_events.sort(key=lambda x: x[0])
    dedup_events = {}
    for ts, ev_type, state in button_events:
        if ts not in dedup_events:
            dedup_events[ts] = state
        else:
            if ev_type == "BUTTON":
                dedup_events[ts] = state
        
    # Handle case where flash does not turn off (auto-cutoff based on dynamic duration of other complete flashes)
    if dedup_events:
        sorted_ts = sorted(dedup_events.keys())
        
        # Find all complete pulse durations (1 followed by 0) to determine standard duration dynamically
        durations = []
        for i in range(len(sorted_ts) - 1):
            ts_curr = sorted_ts[i]
            state_curr = dedup_events[ts_curr]
            if state_curr == 1:
                # Find the next event that is 0
                for j in range(i + 1, len(sorted_ts)):
                    ts_next = sorted_ts[j]
                    state_next = dedup_events[ts_next]
                    if state_next == 0:
                        durations.append(ts_next - ts_curr)
                        break
                    elif state_next == 1:
                        # Another ON event occurred before an OFF event, so this was not completed normally
                        break
                        
        # Determine the standard active duration (median of all completed pulses) to filter out startup noise
        if durations:
            standard_duration = int(np.median(durations))
        else:
            standard_duration = DEFAULT_FLASH_DURATION_MS  # fallback default
                
        # Scan chronologically to reconstruct any orphan OFF events (state 0 when last_state was 0)
        reconstructed_events = {}
        last_state = 0
        for ts in sorted_ts:
            state = dedup_events[ts]
            if state == 1:
                reconstructed_events[ts] = 1
                last_state = 1
            elif state == 0:
                if last_state == 0:
                    # Inferred ON event exactly standard_duration before the OFF event
                    inferred_on_ts = ts - standard_duration
                    reconstructed_events[inferred_on_ts] = 1
                reconstructed_events[ts] = 0
                last_state = 0
                
        dedup_events = reconstructed_events
        sorted_ts = sorted(dedup_events.keys())
        
        # Scan and insert off-events for any on-event that lacks one
        new_events = {}
        i = 0
        while i < len(sorted_ts):
            ts_curr = sorted_ts[i]
            state_curr = dedup_events[ts_curr]
            new_events[ts_curr] = state_curr
            
            if state_curr == 1:
                has_off = False
                if i + 1 < len(sorted_ts):
                    ts_next = sorted_ts[i+1]
                    state_next = dedup_events[ts_next]
                    if state_next == 0 and (ts_next - ts_curr) < (standard_duration * 1.5):
                        has_off = True
                
                if not has_off:
                    art_off_ts = ts_curr + standard_duration
                    new_events[art_off_ts] = 0
            i += 1
            
        dedup_events = new_events
        
    ts_arr = np.array(timestamps, dtype=np.int64)
    pupil_size_arr = np.array(pupil_size, dtype=np.float64)
    
    # Map button/input events to samples to check when blue light is active
    if dedup_events:
        btn_ts_arr = np.array(sorted(dedup_events.keys()), dtype=np.int64)
        btn_states_arr = np.array([dedup_events[ts] for ts in btn_ts_arr], dtype=np.int64)
        
        indices = np.searchsorted(btn_ts_arr, ts_arr, side="right") - 1
        blue_active = np.zeros_like(ts_arr, dtype=np.int64)
        valid_mask = (indices >= 0)
        blue_active[valid_mask] = btn_states_arr[indices[valid_mask]]
    else:
        print("Warning: No BUTTON/INPUT events found in the file. / 警告: ファイルにBUTTON/INPUTイベントが見つかりませんでした。")
        blue_active = np.zeros_like(ts_arr, dtype=np.int64)
        
    # Calculate elapsed time columns
    if len(ts_arr) > 0:
        start_ts = ts_arr[0]
        elapsed_ms = ts_arr - start_ts
        elapsed_sec = elapsed_ms / 1000.0
    else:
        elapsed_ms = np.array([], dtype=np.int64)
        elapsed_sec = np.array([], dtype=np.float64)
        
    df = pd.DataFrame({
        "timestamp": ts_arr,
        "elasped_ms": elapsed_ms,
        "elasped_sec": elapsed_sec,
        "blue_active": blue_active,
        "pupil_size": pupil_size_arr
    })
    
    # Filter saccade events to keep only those from the tracked eye
    saccade_intervals = [(start, end) for eye, start, end in saccade_events if eye == tracked_eye]
    
    # 2. Perform PCHIP Interpolation for Blink & Saccade removal
    # (uses configuration cell values for SLIDING_WINDOW_SIZE and BLINK_THRESHOLD)
    
    df['cleaned_pupil_size'] = df['pupil_size'].astype(float)
    
    # 2a. Blink Masking
    blink_mask = df['cleaned_pupil_size'] < BLINK_THRESHOLD
    expanded_blink_mask = blink_mask.rolling(window=SLIDING_WINDOW_SIZE, center=True, min_periods=1).max().astype(bool)
    
    # 2b. Saccade Masking
    saccade_mask = np.zeros(len(df), dtype=bool)
    for start, end in saccade_intervals:
        # Mask window: start - 20 ms to end + 60 ms
        cond = (df['timestamp'] >= (start - SACCADE_PRE_PAD_MS)) & (df['timestamp'] <= (end + SACCADE_POST_PAD_MS))
        saccade_mask = saccade_mask | cond.values
        
    # Combine masks
    combined_mask = expanded_blink_mask | saccade_mask
    df.loc[combined_mask, 'cleaned_pupil_size'] = np.nan
    df['cleaned_pupil_size'] = df['cleaned_pupil_size'].interpolate(method='pchip', limit_area='inside')
    df['cleaned_pupil_size'] = df['cleaned_pupil_size'].ffill().bfill()
    
    print(f"\nProcessing completed successfully! / 処理が正常に完了しました！")
    print(f"Total parsed samples / 総サンプル数: {len(df)}")
    print(f"Cleaned {blink_mask.sum()} raw blink samples (expanded to {expanded_blink_mask.sum()} samples for transient edges) and masked {saccade_mask.sum()} samples during saccades.")
    print(f"瞬き生データ {blink_mask.sum()} サンプルをクリーニング（過渡期を含め計 {expanded_blink_mask.sum()} サンプルを補間）、およびサッケード中の {saccade_mask.sum()} サンプルをマスク。")
    
    return df

In [ ]:
# Run parser on data
if IN_COLAB:
    file_data = io.BytesIO(uploaded[asc_filename])
    df_clean = parse_and_clean_asc(file_data, asc_filename)
else:
    with open(asc_filename, "rb") as f:
        df_clean = parse_and_clean_asc(f, asc_filename)

df_clean.head()


## Step 3a: Visualize the Raw Uncleaned Data
## ステップ 3a: 生データ（未クリーニング）の視覚化

Here is a plot of the raw pupil size data showing the signal drops to zero during blink artifacts.

瞬き中に信号がゼロに急降下している生の瞳孔径データのグラフです。

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'IPAexGothic'
plt.figure(figsize=(14, 7))

# Plot raw pupil size only
plt.plot(df_clean['elasped_sec'], df_clean['pupil_size'], label='Raw Pupil Size (生データ)', color='darkred', linewidth=1)

# Shaded region for active blue LED phase
transitions = df_clean['blue_active'].diff().fillna(0)
starts = df_clean.loc[transitions == 1, 'elasped_sec'].values
ends = df_clean.loc[transitions == -1, 'elasped_sec'].values
if len(starts) > len(ends):
    ends = np.append(ends, df_clean['elasped_sec'].max())

for start, end in zip(starts, ends):
    plt.axvspan(start, end, color='blue', alpha=0.1, label='Blue LED Active (青色LEDオン)' if start == starts[0] else "")

plt.title(f"Raw Pupil Size over Time (File: {os.path.basename(asc_filename)}) / 生の瞳孔径の経時変化", fontsize=14, fontweight='bold')
plt.xlabel("Elapsed Time (seconds) / 経過時間 (秒)", fontsize=12)
plt.ylabel("Pupil Size (Raw) / 生の瞳孔径 (直径)", fontsize=12)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Step 3b: Visualize the Interpolation Quality (Raw vs. Cleaned Overlay)
## ステップ 3b: 補間クオリティの視覚化（生データとクリーニングデータの重ね合わせ）

Here is the comparison plot showing how the PCHIP interpolation has bridged the blink gaps without ripples or overshoot.

PCHIP補間によって、波打ちやオーバーシュートを起こさずに瞬きの隙間がどのように滑らかに繋がったかを示す比較プロットです。

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'IPAexGothic'
plt.figure(figsize=(14, 7))

# 1. Plot raw pupil size
plt.plot(df_clean['elasped_sec'], df_clean['pupil_size'], label='Raw Pupil Size (生データ)', color='lightgray', alpha=0.8, linewidth=1)

# 2. Plot PCHIP cleaned pupil size
plt.plot(df_clean['elasped_sec'], df_clean['cleaned_pupil_size'], label='Cleaned Pupil Size (PCHIP補間)', color='darkgreen', linewidth=1.5)

# 3. Shaded region for active blue LED phase
transitions = df_clean['blue_active'].diff().fillna(0)
starts = df_clean.loc[transitions == 1, 'elasped_sec'].values
ends = df_clean.loc[transitions == -1, 'elasped_sec'].values
if len(starts) > len(ends):
    ends = np.append(ends, df_clean['elasped_sec'].max())

for start, end in zip(starts, ends):
    plt.axvspan(start, end, color='blue', alpha=0.1, label='Blue LED Active (青色LEDオン)' if start == starts[0] else "")

plt.title(f"Raw vs Cleaned Pupil Size (File: {os.path.basename(asc_filename)}) / 生データと補間データの比較", fontsize=14, fontweight='bold')
plt.xlabel("Elapsed Time (seconds) / 経過時間 (秒)", fontsize=12)
plt.ylabel("Pupil Size / 瞳孔径 (直径)", fontsize=12)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Step 4: Export and Download Cleaned CSV & Excel Files
## ステップ 4: クリーニング済みのCSV/Excelファイルのエクスポートとダウンロード

In [ ]:
base_name = os.path.splitext(os.path.basename(asc_filename))[0]

# 1. Export raw parsed data (from first pipeline) to both CSV and Excel
df_raw = df_clean.drop(columns=['cleaned_pupil_size'])
output_raw_csv = base_name + "_parsed.csv"
output_raw_xlsx = base_name + "_parsed.xlsx"
df_raw.to_csv(output_raw_csv, index=False)
df_raw.to_excel(output_raw_xlsx, index=False)
print(f"Saved parsed raw data to: {output_raw_csv} and {output_raw_xlsx}")
print(f"解析生データを保存しました: {output_raw_csv} および {output_raw_xlsx}")

# 2. Export cleaned data with PCHIP interpolation (from second pipeline) to both CSV and Excel
output_clean_csv = base_name + "_cleaned.csv"
output_clean_xlsx = base_name + "_cleaned.xlsx"
df_clean.to_csv(output_clean_csv, index=False)
df_clean.to_excel(output_clean_xlsx, index=False)
print(f"Saved cleaned PCHIP data to: {output_clean_csv} and {output_clean_xlsx}")
print(f"解析＆補間したデータを保存しました: {output_clean_csv} および {output_clean_xlsx}")

if IN_COLAB:
    print("\nDownloading files to your machine... / ファイルをパソコンへダウンロード中...")
    files.download(output_raw_csv)
    files.download(output_raw_xlsx)
    files.download(output_clean_csv)
    files.download(output_clean_xlsx)
else:
    print(f"\nFiles exported locally to the current directory: {os.getcwd()}")
    print(f"ファイルはローカルのカレントディレクトリにエクスポートされました: {os.getcwd()}")
